# 08 — khive Integration

lionag2 optionally integrates with khive for persistent knowledge:

- **`KhiveKnowledgeStore`** — implements AG2's `KnowledgeStore` protocol backed by khive memory. AG2's full harness (working memory, episodic memory, compaction, aggregation) runs on khive natively.
- **`KhiveToolkit`** — AG2 `Toolkit` wrapping khive SDK for explicit memory/graph/messaging calls.

Without khive, the engine falls back to `MemoryKnowledgeStore` (in-memory). With khive, each research run **compounds** — future recall gets better because the knowledge graph grows.

In [ ]:
from lionag2.tools import KhiveToolkit, khive_available

print(f"khive SDK available: {khive_available()}")

## KhiveKnowledgeStore

AG2's `KnowledgeStore` protocol is a flat virtual filesystem: `read(path)`, `write(path, content)`, `list(path)`, `delete(path)`. khive's memory is key-value, not path-based — the adapter uses a `kstore:` prefix convention:

```
write("/memory/working.md", content)
  → khive.memory.remember("kstore:/memory/working.md\n{content}")

read("/memory/working.md")
  → khive.memory.recall("kstore:/memory/working.md") → strip prefix
```

The engine picks the store based on environment:

```python
if self._has_khive:
    self._knowledge_store = KhiveKnowledgeStore(
        api_key=khive_api_key,
        namespace=khive_namespace,
    )
else:
    self._knowledge_store = MemoryKnowledgeStore()
```

In [ ]:
import inspect
from lionag2.tools import KhiveKnowledgeStore

# KhiveKnowledgeStore implements AG2's Storage protocol
print(inspect.getsource(KhiveKnowledgeStore))

## KhiveToolkit

While `KhiveKnowledgeStore` runs AG2's harness on khive **implicitly**, the `KhiveToolkit` gives agents **explicit** tools for memory, graph, and messaging:

| Tool | Description |
|---|---|
| `memory_recall` | Search persistent memory for past findings |
| `memory_remember` | Save a finding to persistent memory |
| `graph_search` | Search knowledge graph |
| `graph_add_entity` | Create a paper/dataset/concept entity |
| `graph_add_link` | Link two entities (cites, contradicts, etc.) |
| `graph_neighbors` | Get entities connected to a given entity |
| `send_message` | Send a message to another team |
| `list_messages` | Check inbox for cross-team messages |

Each agent gets a **fresh** toolkit instance (fresh `AsyncKhive` client) to avoid connection sharing issues.

In [ ]:
if khive_available():
    kt = KhiveToolkit(namespace="demo")
    for t in kt.tools:
        print(f"  {t.name}: {t.schema.function.description[:60]}")
else:
    print("khive SDK not installed — toolkit not available.")
    print("Install with: uv add 'lionag2[khive]'")

## The Connector agent

The Connector only joins the roster when khive is available — native AG2 has no graph.

```python
if self._has_khive:
    roster.append(CONNECTOR)
```

Its job is to weave discoveries into the knowledge graph after the research team finishes a node: create entities for papers/datasets/methods, link them with typed relations (`cites`, `contradicts`, `uses_dataset`), and save one-line distillations to memory.

This is the **compounding quality loop**: each research run builds the graph, future `memory_recall` and `graph_search` calls surface richer context.

## Up next

After all exploration nodes finish, findings go through cross-checking and iterative paper writing. Tutorial 09 covers both.